# Proyecto MLOps: Análisis de MiBici

## Objetivo
Desarrollar un pipeline reproducible de ingesta, limpieza y preparación
de datos para resolver tres problemas de Machine Learning:

- P1: Predicción de demanda de viajes (regresión).
- P2: Detección de presión operativa (clasificación).
- P3: Agrupamiento de estaciones según patrones de uso (clustering).

## Periodo inicial
Enero a mayo de 2026.

## Fuente
[Portal de datos abiertos MiBici](https://mibici.net/es/datos-abiertos/)

## Consideraciones
Los datos de viajes no incluyen el inventario real de bicicletas por
estación. Por ello, P2 utilizará inicialmente una variable proxy de riesgo.

In [1]:
import pandas as pd

csv_files = ['datos_abiertos_2026_01.csv',
             'datos_abiertos_2026_02.csv',
             'datos_abiertos_2026_03.csv',
             'datos_abiertos_2026_04.csv',
             'datos_abiertos_2026_05.csv']

list_dfs = []
for file in csv_files:
    df_temp = pd.read_csv(f"/content/{file}", encoding='latin-1')
    # Corregir errores comunes de codificación en los nombres de las columnas
    df_temp.columns = df_temp.columns.str.replace('AÃ±o_de_nacimiento', 'Año_de_nacimiento')
    df_temp.columns = df_temp.columns.str.replace('Año_de_nacimiento', 'Año_de_nacimiento') # Asegurar consistencia
    list_dfs.append(df_temp)

# Unir todos los DataFrames corregidos
df_mibici = pd.concat(list_dfs, ignore_index=True)

print(f"DataFrame unificado correctamente con {df_mibici.shape[0]} filas y {df_mibici.shape[1]} columnas.")
print("Columnas resultantes:", df_mibici.columns.tolist())
display(df_mibici.head())

DataFrame unificado correctamente con 1734925 filas y 8 columnas.
Columnas resultantes: ['Viaje_Id', 'Usuario_Id', 'Genero', 'Año_de_nacimiento', 'Inicio_del_viaje', 'Fin_del_viaje', 'Origen_Id', 'Destino_Id']


,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id
0,42349943,1726166,M,1967.0,2026-01-01 00:03:07,2026-01-01 00:03:39,64,64
1,42349944,1726166,M,1967.0,2026-01-01 00:04:05,2026-01-01 00:13:20,64,35
2,42349945,667524,M,1994.0,2026-01-01 00:05:24,2026-01-01 00:33:40,345,70
3,42349946,534776,M,1978.0,2026-01-01 00:05:48,2026-01-01 00:30:54,35,333
4,42349947,161552,M,1997.0,2026-01-01 00:07:07,2026-01-01 00:14:56,268,226


###Verificar el nuevo DataFrame

In [8]:
#Convertir inicio y fin de viaje a tiempo
# Conversión de tipos de datos de fecha
df_mibici["Inicio_del_viaje"] = pd.to_datetime(df_mibici["Inicio_del_viaje"])
df_mibici["Fin_del_viaje"] = pd.to_datetime(df_mibici["Fin_del_viaje"])




Columnas con ID convertidas a categóricas: ['Viaje_Id', 'Usuario_Id', 'Origen_Id', 'Destino_Id']


In [9]:
df_mibici.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1734925 entries, 0 to 1734924
Data columns (total 8 columns):
 #   Column             Dtype         
---  ------             -----         
 0   Viaje_Id           object        
 1   Usuario_Id         object        
 2   Genero             object        
 3   Año_de_nacimiento  float64       
 4   Inicio_del_viaje   datetime64[ns]
 5   Fin_del_viaje      datetime64[ns]
 6   Origen_Id          object        
 7   Destino_Id         object        
dtypes: datetime64[ns](2), float64(1), object(5)
memory usage: 105.9+ MB


## Fase de Limpieza de Datos, Estadísticas Descriptivas y Detección de Outliers

En esta sección realizaremos un diagnóstico profundo del dataset unificado `df_mibici` para asegurar la calidad de los datos antes de construir los modelos.
```

In [13]:
import numpy as np

#Convertir a categórica/string cualquier columna que contenga "Id" o "ID" en su nombre
id_cols = [col for col in df_mibici.columns if "Id" in col or "ID" in col]
df_mibici[id_cols] = df_mibici[id_cols].astype(str)

print("Columnas con ID convertidas a categóricas:", id_cols)

# 2. Separación de las variables numéricas y de categoría para obtener estadísticas
bici_num = df_mibici.select_dtypes(include=np.number)
bici_cat = df_mibici.select_dtypes(exclude=np.number)

# 3. Mostrar las columnas en cada grupo
print("\nVariables numéricas:", list(bici_num.columns))
print("Variables no numéricas / categóricas:", list(bici_cat.columns))

Columnas con ID convertidas a categóricas: ['Viaje_Id', 'Usuario_Id', 'Origen_Id', 'Destino_Id']

Variables numéricas: ['Año_de_nacimiento']
Variables no numéricas / categóricas: ['Viaje_Id', 'Usuario_Id', 'Genero', 'Inicio_del_viaje', 'Fin_del_viaje', 'Origen_Id', 'Destino_Id']


In [14]:
# 2. Diagnóstico y Limpieza de Valores Nulos y Duplicados
print("--- Valores Nulos por Columna ---")
nulos = df_mibici.isnull().sum()
porcentaje_nulos = (nulos / len(df_mibici)) * 100
resumen_nulos = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': porcentaje_nulos})
display(resumen_nulos)

# Eliminar duplicados si existen
duplicados = df_mibici.duplicated().sum()
print(f"\nRegistros completamente duplicados encontrados: {duplicados}")
if duplicados > 0:
    df_mibici = df_mibici.drop_duplicates()
    print("Duplicados eliminados.")

# Imputación/Tratamiento de Nulos
# El género faltante se puede catalogar como 'NE' (No Especificado)
df_mibici['Genero'] = df_mibici['Genero'].fillna('NE')

# Para Año de nacimiento, imputaremos temporalmente con la mediana para no perder registros
mediana_nacimiento = df_mibici['Año_de_nacimiento'].median()
df_mibici['Año_de_nacimiento'] = df_mibici['Año_de_nacimiento'].fillna(mediana_nacimiento)

print("\nValores nulos después del tratamiento:")
print(df_mibici.isnull().sum())

--- Valores Nulos por Columna ---


,Nulos,Porcentaje (%)
Viaje_Id,0,0.000000
Usuario_Id,0,0.000000
Genero,11393,0.656685
Año_de_nacimiento,200,0.011528
Inicio_del_viaje,0,0.000000
Fin_del_viaje,0,0.000000
Origen_Id,0,0.000000
Destino_Id,0,0.000000



Registros completamente duplicados encontrados: 0

Valores nulos después del tratamiento:
Viaje_Id             0
Usuario_Id           0
Genero               0
Año_de_nacimiento    0
Inicio_del_viaje     0
Fin_del_viaje        0
Origen_Id            0
Destino_Id           0
dtype: int64


### 3. Análisis de Outliers (Valores Atípicos)
Analizaremos dos variables críticas:
1. **Edad**: Usuarios con edades excesivamente avanzadas o negativas.
2. **Duración del Viaje**: Viajes extremadamente cortos (segundos) o excesivamente largos (días, que suelen indicar fallas de anclaje).

In [15]:
# Calcular Edad y Duración del Viaje en minutos
df_mibici['Edad'] = 2026 - df_mibici['Año_de_nacimiento']
df_mibici['Duracion_Minutos'] = (df_mibici['Fin_del_viaje'] - df_mibici['Inicio_del_viaje']).dt.total_seconds() / 60.0

# Identificar Outliers en Edad
print("--- Distribución de Edad ---")
display(df_mibici['Edad'].describe(percentiles=[0.01, 0.05, 0.95, 0.99]))

# Identificar Outliers en Duración
print("\n--- Distribución de Duración de Viajes (Minutos) ---")
display(df_mibici['Duracion_Minutos'].describe(percentiles=[0.01, 0.05, 0.95, 0.99]))

--- Distribución de Edad ---


,Edad
count,1.734925e+06
mean,3.557099e+01
std,1.959868e+01
min,1.600000e+01
1%,1.900000e+01
5%,2.200000e+01
50%,3.300000e+01
95%,5.800000e+01
99%,6.700000e+01
max,1.827000e+03



--- Distribución de Duración de Viajes (Minutos) ---


,Duracion_Minutos
count,1.734925e+06
mean,1.116609e+01
std,1.013219e+01
min,0.000000e+00
1%,1.833333e-01
5%,2.116667e+00
50%,9.550000e+00
95%,2.468333e+01
99%,3.366667e+01
max,5.920050e+03


In [16]:
# Filtrado de Outliers aplicando reglas de negocio lógicas:
# - Edad: Entre 16 y 90 años (Límite operacional realista)
# - Duración: Entre 1 minuto y 180 minutos (Viajes menores a 1 min suelen ser errores de anclaje, mayores a 3 horas son anomalías o robos)

filtros_logicos = (
    (df_mibici['Edad'] >= 16) & (df_mibici['Edad'] <= 90) &
    (df_mibici['Duracion_Minutos'] >= 1.0) & (df_mibici['Duracion_Minutos'] <= 180.0)
)

antes_filtro = len(df_mibici)
df_mibici_clean = df_mibici[filtros_logicos].copy()
despues_filtro = len(df_mibici_clean)

print(f"Filas antes de la limpieza de outliers: {antes_filtro}")
print(f"Filas después de remover outliers: {despues_filtro}")
print(f"Registros removidos: {antes_filtro - despues_filtro} ({((antes_filtro - despues_filtro)/antes_filtro)*100:.2f}% del total)")

Filas antes de la limpieza de outliers: 1734925
Filas después de remover outliers: 1677043
Registros removidos: 57882 (3.34% del total)


### Construcción de Variables para P1: Predicción de Demanda
Agrupamos la cantidad de viajes salientes por estación de origen (`Origen_Id`) y por hora.

In [17]:
# Extraer hora y fecha para el análisis
df_mibici['Fecha_Inicio'] = df_mibici['Inicio_del_viaje'].dt.date
df_mibici['Hora_Inicio'] = df_mibici['Inicio_del_viaje'].dt.hour

#Calcular Edad basada en el año actual (2026) y la columna 'Año_de_nacimiento'
df_mibici['Edad'] = 2026 - df_mibici['Año_de_nacimiento']

# Agrupar viajes salientes (Demanda)
p1_demanda = df_mibici.groupby(['Origen_Id', 'Fecha_Inicio', 'Hora_Inicio', 'Edad']).size().reset_index(name='Cantidad_Viajes_Salientes')

print("Ejemplo de variables para P1 (Demanda):")
display(p1_demanda.head())

Ejemplo de variables para P1 (Demanda):


,Origen_Id,Fecha_Inicio,Hora_Inicio,Edad,Cantidad_Viajes_Salientes
0,10,2026-01-01,0,37.0,1
1,10,2026-01-01,7,29.0,1
2,10,2026-01-01,10,37.0,1
3,10,2026-01-01,14,52.0,1
4,10,2026-01-01,15,29.0,1


### Construcción de Variables para P2: Saturación / Presión Operativa (Proxy)
Calculamos el balance neto (Llegadas - Salidas) acumulado por estación y hora para medir la presión en la operación.

In [18]:
# Extraer fecha y hora para los viajes finalizados
df_mibici['Fecha_Fin'] = df_mibici['Fin_del_viaje'].dt.date
df_mibici['Hora_Fin'] = df_mibici['Fin_del_viaje'].dt.hour

# Contar salidas (salidas por origen, fecha y hora)
salidas = df_mibici.groupby(['Origen_Id', 'Fecha_Inicio', 'Hora_Inicio']).size().reset_index(name='Salidas')
salidas.columns = ['Estacion_Id', 'Fecha', 'Hora', 'Salidas']

# Contar entradas (llegadas por destino, fecha y hora)
entradas = df_mibici.groupby(['Destino_Id', 'Fecha_Fin', 'Hora_Fin']).size().reset_index(name='Entradas')
entradas.columns = ['Estacion_Id', 'Fecha', 'Hora', 'Entradas']

# Unir flujos de entradas y salidas
flujo_estaciones = pd.merge(entradas, salidas, on=['Estacion_Id', 'Fecha', 'Hora'], how='outer').fillna(0)

# Calcular balance neto (Entradas - Salidas)
flujo_estaciones['Balance_Neto'] = flujo_estaciones['Entradas'] - flujo_estaciones['Salidas']

print("Ejemplo de variables para P2 (Balance Neto / Presión Operativa):")
display(flujo_estaciones.head())

Ejemplo de variables para P2 (Balance Neto / Presión Operativa):


,Estacion_Id,Fecha,Hora,Entradas,Salidas,Balance_Neto
0,10,2026-01-01,0,0.0,1.0,-1.0
1,10,2026-01-01,7,0.0,1.0,-1.0
2,10,2026-01-01,10,0.0,1.0,-1.0
3,10,2026-01-01,11,1.0,0.0,1.0
4,10,2026-01-01,14,0.0,1.0,-1.0


### Construcción de la Variable Binaria para P2: Riesgo de Capacidad (t+1)
Creamos un proxy de estado crítico basado en extremos del balance acumulado diario por estación y lo desplazamos una hora en el futuro para definir nuestra variable a predecir.

In [19]:
# 1. Ordenar datos cronológicamente por estación
flujo_estaciones = flujo_estaciones.sort_values(by=['Estacion_Id', 'Fecha', 'Hora']).reset_index(drop=True)

# 2. Calcular el balance acumulado en el día para estimar el llenado/vaciado relativo
flujo_estaciones['Balance_Acumulado_Dia'] = flujo_estaciones.groupby(['Estacion_Id', 'Fecha'])['Balance_Neto'].cumsum()

# 3. Definir límites proxy de riesgo basados en los percentiles de la propia estación
# (Estaciones con balances acumulados muy negativos están en riesgo de desabasto; muy positivos en saturación)
limites = flujo_estaciones.groupby('Estacion_Id')['Balance_Acumulado_Dia'].agg([lambda x: x.quantile(0.1), lambda x: x.quantile(0.9)]).reset_index()
limites.columns = ['Estacion_Id', 'Umbral_Desabasto', 'Umbral_Saturacion']

flujo_con_umbrales = pd.merge(flujo_estaciones, limites, on='Estacion_Id', how='left')

# 4. Crear la variable de estado actual (1: Riesgo por desabasto/saturación, 0: Normal)
flujo_con_umbrales['Estado_Critico'] = ((
    (flujo_con_umbrales['Balance_Acumulado_Dia'] <= flujo_con_umbrales['Umbral_Desabasto']) |
    (flujo_con_umbrales['Balance_Acumulado_Dia'] >= flujo_con_umbrales['Umbral_Saturacion'])
) & (flujo_con_umbrales['Balance_Neto'] != 0)).astype(int)

# 5. Crear la variable objetivo (Target) para la Próxima Hora (t+1)
# Desplazamos (shift) el estado crítico una hora hacia atrás por estación para que represente el riesgo en t+1
flujo_con_umbrales['Target_Riesgo_Proxima_Hora'] = flujo_con_umbrales.groupby(['Estacion_Id', 'Fecha'])['Estado_Critico'].shift(-1)

# Eliminar filas con NaN en el target debido al desplazamiento al final del día
p2_clasificacion = flujo_con_umbrales.dropna(subset=['Target_Riesgo_Proxima_Hora']).copy()
p2_clasificacion['Target_Riesgo_Proxima_Hora'] = p2_clasificacion['Target_Riesgo_Proxima_Hora'].astype(int)

print("Distribución de la variable objetivo (0: Normal, 1: Riesgo en t+1):")
print(p2_clasificacion['Target_Riesgo_Proxima_Hora'].value_counts(normalize=True) * 100)

print("\nEjemplo de variables preparadas para P2:")
display(p2_clasificacion[['Estacion_Id', 'Fecha', 'Hora', 'Balance_Neto', 'Balance_Acumulado_Dia', 'Estado_Critico', 'Target_Riesgo_Proxima_Hora']].head(10))

Distribución de la variable objetivo (0: Normal, 1: Riesgo en t+1):
Target_Riesgo_Proxima_Hora
0    78.055903
1    21.944097
Name: proportion, dtype: float64

Ejemplo de variables preparadas para P2:


,Estacion_Id,Fecha,Hora,Balance_Neto,Balance_Acumulado_Dia,Estado_Critico,Target_Riesgo_Proxima_Hora
0,10,2026-01-01,0,-1.0,-1.0,0,0
1,10,2026-01-01,7,-1.0,-2.0,0,0
2,10,2026-01-01,10,-1.0,-3.0,0,0
3,10,2026-01-01,11,1.0,-2.0,0,0
4,10,2026-01-01,14,-1.0,-3.0,0,0
5,10,2026-01-01,15,-1.0,-4.0,0,0
7,10,2026-01-02,6,-1.0,-1.0,0,0
8,10,2026-01-02,7,0.0,-1.0,0,0
9,10,2026-01-02,8,-2.0,-3.0,0,0
10,10,2026-01-02,9,-2.0,-5.0,0,0


### Construcción de Variables para P3: Agrupamiento (Clustering) de Estaciones
Determinamos el perfil de uso de cada estación calculando el porcentaje promedio de viajes que ocurren en cada una de las 24 horas del día.

In [20]:
# Crear una tabla cruzada de Estación de Origen vs Hora de Inicio
perfil_horas = pd.crosstab(df_mibici['Origen_Id'], df_mibici['Hora_Inicio'])

# Convertir las frecuencias absolutas a proporciones (porcentajes) por fila
perfil_horas_porcentaje = perfil_horas.div(perfil_horas.sum(axis=1), axis=0) * 100

# Renombrar columnas para claridad
perfil_horas_porcentaje.columns = [f'Pct_Viajes_Hora_{h}' for h in perfil_horas_porcentaje.columns]

print("Perfil horario de las estaciones para P3 (Clustering):")
display(perfil_horas_porcentaje.head())

Perfil horario de las estaciones para P3 (Clustering):


,Pct_Viajes_Hora_0,Pct_Viajes_Hora_1,Pct_Viajes_Hora_5,Pct_Viajes_Hora_6,Pct_Viajes_Hora_7,Pct_Viajes_Hora_8,Pct_Viajes_Hora_9,Pct_Viajes_Hora_10,Pct_Viajes_Hora_11,Pct_Viajes_Hora_12,...,Pct_Viajes_Hora_14,Pct_Viajes_Hora_15,Pct_Viajes_Hora_16,Pct_Viajes_Hora_17,Pct_Viajes_Hora_18,Pct_Viajes_Hora_19,Pct_Viajes_Hora_20,Pct_Viajes_Hora_21,Pct_Viajes_Hora_22,Pct_Viajes_Hora_23
Origen_Id,,,,,,,,,,,,,,,,,,,,,
10,0.274927,0.0,0.476540,5.571848,8.541056,8.962610,5.608504,4.490469,4.343842,6.359971,...,6.341642,5.168622,4.600440,8.174487,7.294721,6.433284,5.333578,3.115836,1.502933,0.531525
100,0.083333,0.0,0.250000,10.500000,5.833333,8.666667,10.250000,2.916667,4.500000,7.416667,...,5.583333,7.333333,6.916667,9.333333,5.333333,4.000000,3.583333,2.083333,0.583333,0.250000
101,0.405954,0.0,3.788904,4.330176,2.841678,8.660352,10.284168,10.690122,7.036536,4.059540,...,3.924222,7.171854,5.818674,7.577808,6.901218,5.006766,3.518268,2.029770,1.488498,1.623816
102,0.041127,0.0,0.514086,2.426486,4.894098,9.685379,6.086778,5.613819,5.819453,5.634382,...,6.929879,7.834670,5.737199,6.744808,6.189595,6.498046,5.099733,4.441703,2.364795,0.966482
103,0.578339,0.0,0.052576,0.473186,3.575184,10.410095,13.354364,5.573081,5.099895,3.995794,...,4.416404,12.145110,10.778128,9.043113,4.889590,2.628812,3.627760,2.208202,1.787592,1.261830
